# 1. GPT-2
- **모델 이름**: `"gpt2"` (= GPT-2 Small, 117M)
- **파라미터 수**: 117M (1억 1,700만 개)
- **레이어 수**: 12 Transformer blocks
- **Hidden size**: 768
- **Attention heads**: 12개
- **FFN 내부 차원**: 3072

## 학습 데이터 정보
- **학습 데이터 용량**: 약 40GB 텍스트
- **총 토큰 수**: 약 10B tokens (100억 토큰)
- **사용 데이터셋**: **WebText**
  - Reddit에서 3개 이상 upvote를 받은 링크 기반 수집 데이터
  - 고품질 웹 문서 중심

## 최대 시퀀스 길이 (Context Length)
- **GPT-2 Small 최대 길이**: 1024 tokens
- 입력 + 생성 토큰을 **합쳐서** 최대 1024
- 1024를 초과하면:
  - 그 이상은 positional embedding이 없음

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model_name = "gpt2"   # 117M
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

model.eval()

prompt = "The future of artificial intelligence is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.9
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/Users/pc/miniforge3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: 
AutoModelForCausalLM requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [1]:
while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    inputs = tokenizer(user_input, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7
            top_p=0.8
            repetition_penalty=1.3
        )

    print("GPT:", tokenizer.decode(outputs[0], skip_special_tokens=True))

SyntaxError: invalid syntax. Perhaps you forgot a comma? (374166883.py, line 13)

## GPT-2 실행 시 내부 동작 과정 (Inference Flow)
1. **토큰화 (Tokenization)**
   - 입력 문장을 BPE 기반 토큰으로 변환
   - 보통 문장 하나 ≈ 10~20 tokens

2. **Embedding Lookup**
   - 각 토큰 ID → 768차원 벡터로 변환
   - Positional embedding 더해짐

3. **Transformer Block 통과 (12개)**
   - Multi-Head Self-Attention
   - Feed Forward Network (FFN)
   - Residual + LayerNorm
   - 위 과정이 12번 반복됨

4. **마지막 Linear Layer**
   - 768 → 50257 차원으로 projection
   - vocab 전체에 대한 logits 생성

5. **Softmax**
   - 50257개 토큰에 대한 확률 분포 계산

6. **Sampling**
   - argmax / temperature / top-k / top-p 방식으로
   - 다음 토큰 1개 선택

7. **반복**
   - 새로 생성된 토큰을 입력에 추가
   - 최대 길이(1024)까지 반복